In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from scipy.optimize import minimize
import matplotlib.gridspec as gridspec


# Neglecting speed distribution

In [ ]:
def research(x):
    """
        Grows logarithmically from 0 to 200k as x goes from 0 to 100
        x: int, the research percentage (0 to 100)
    """
    return 200_000 * np.log(1 + x) / np.log(1 + 100)

def scale(x):
    """
        Grows linearly from 0 to 7 as x goes from 0 to 100

        x: int, the scale percentage (0 to 100)
    """
    return 7 * x / 100

def speed_direct_mapping(x):
    """
    Linear proxy (rank-based to be implemented later).
    """
    return 0.1 + 0.8 * x / 100

def speed_real(x):
    """
    Rank-based across all players:
    - Highest speed investment receives a `0.9` multiplier.
    - Lowest receives `0.1`.
    - Everyone in between is scaled linearly by rank, equal investments share the same rank.
    """
    pass

BUDGET = 50_000

def PnL(r_pct, sc_pct, sp_pct):
    """
    r_pct, sc_pct, sp_pct: floats in [0, 100], must sum to <= 100.
    Budget used = 50_000 * (total allocation as a fraction).
    """
    total_alloc = r_pct + sc_pct + sp_pct          # in [0, 300], constrained to ≤ 100
    budget_used = BUDGET * (total_alloc / 100)
    gross = research(r_pct) * scale(sc_pct) * speed_direct_mapping(sp_pct)
    return gross - budget_used
    

In [11]:
r_pct, sc_pct, sp_pct = 40, 10, 40

pnl = PnL(r_pct, sc_pct, sp_pct)
print(round(pnl, 2))

2313.62


In [14]:


def neg_pnl(x):
    """Objective for scipy (minimizes, so we negate)."""
    return -PnL(*x)

result = minimize(
    neg_pnl,
    x0=[40, 20, 40],                               # starting guess
    method="SLSQP",
    bounds=[(0, 100), (0, 100), (0, 100)],          # each pillar in [0, 100]
    constraints=[
        {"type": "ineq", "fun": lambda x: 100 - sum(x)}   # sum ≤ 100 (not =)
    ],
    options={"ftol": 1e-12, "maxiter": 10_000}
)

r_opt, sc_opt, sp_opt = result.x
total_opt = sum(result.x)
budget_used  = BUDGET * (total_opt / 100)
budget_saved = BUDGET - budget_used

print(f"Research : {r_opt:.2f}%")
print(f"Scale    : {sc_opt:.2f}%")
print(f"Speed    : {sp_opt:.2f}%")
print(f"Total    : {total_opt:.2f}%  (budget saved: {budget_saved:,.0f} XIRECs)")
print(f"Net PnL  : {-result.fun:,.0f}")

Research : 16.02%
Scale    : 48.24%
Speed    : 35.74%
Total    : 100.00%  (budget saved: -0 XIRECs)
Net PnL  : 110,070


In [15]:
np.random.seed(42)
N = 500
results = []

for _ in range(N):
    a, b, c = np.random.dirichlet([1,1,1]) * np.random.uniform(0, 100)
    res = minimize(
        neg_pnl, [a, b, c],
        method="SLSQP",
        bounds=[(0,100),(0,100),(0,100)],
        constraints=[{"type":"ineq","fun": lambda x: 100 - sum(x)}],
        options={"ftol":1e-12,"maxiter":10_000}
    )
    if res.success or res.fun < 0:
        r, sc, sp = res.x
        results.append({"r": r, "sc": sc, "sp": sp, "pnl": -res.fun, "total": r+sc+sp})

df = pd.DataFrame(results)
is_zero = df["total"] < 1
is_opt  = ~is_zero

# --- 4-panel stability figure ---
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Research % across runs",
        "Scale % across runs",
        "Speed % across runs",
        "Net PnL distribution",
    ]
)

idx = list(range(len(df)))

for col_name, row, col, title in [("r",1,1,"Research %"), ("sc",1,2,"Scale %"), ("sp",2,1,"Speed %")]:
    fig.add_trace(go.Scatter(
        x=[i for i in idx if is_opt.iloc[i]],
        y=df[col_name][is_opt].values,
        mode="markers", marker=dict(color="#1D9E75", size=4, opacity=0.5),
        name="converged", legendgroup="converged", showlegend=(col_name=="r")
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=[i for i in idx if is_zero.iloc[i]],
        y=df[col_name][is_zero].values,
        mode="markers", marker=dict(color="#E24B4A", size=4, opacity=0.7),
        name="trivial (zero)", legendgroup="zero", showlegend=(col_name=="r")
    ), row=row, col=col)

# PnL histogram
fig.add_trace(go.Histogram(
    x=df["pnl"][is_opt],
    nbinsx=40, marker_color="#1D9E75", opacity=0.8,
    name="PnL (converged)", showlegend=False
), row=2, col=2)
fig.add_trace(go.Histogram(
    x=df["pnl"][is_zero],
    nbinsx=5, marker_color="#E24B4A", opacity=0.8,
    name="PnL (trivial)", showlegend=False
), row=2, col=2)

fig.update_layout(
    title="Optimizer stability — 500 random restarts",
    height=700, barmode="overlay",
    legend=dict(orientation="h", y=-0.08)
)
fig.show()


In [23]:
r_opt, sc_opt, sp_opt = 16.02, 48.24, 35.74
grid = np.linspace(0, 100, 200)

fig = make_subplots(
    rows=1, cols=3,
    specs=[[{"type":"surface"}, {"type":"surface"}, {"type":"surface"}]],
    subplot_titles=[
        f"PnL(R, SC)  — speed fixed at {sp_opt}%",
        f"PnL(R, SP)  — scale fixed at {sc_opt}%",
        f"PnL(SC, SP) — research fixed at {r_opt}%",
    ]
)

# panel 1: sweep R and SC, fix speed
R, SC = np.meshgrid(grid, grid)
mask = R + SC <= 100
Z1 = np.where(mask, PnL(R, SC, sp_opt), np.nan)

fig.add_trace(go.Surface(x=grid, y=grid, z=Z1,
    colorscale="Viridis", showscale=False, name="R vs SC"), row=1, col=1)

# panel 2: sweep R and SP, fix scale
R, SP = np.meshgrid(grid, grid)
mask = R + SP <= 100
Z2 = np.where(mask, PnL(R, sc_opt, SP), np.nan)

fig.add_trace(go.Surface(x=grid, y=grid, z=Z2,
    colorscale="RdYlBu", showscale=False, name="R vs SP"), row=1, col=2)

# panel 3: sweep SC and SP, fix research
SC, SP = np.meshgrid(grid, grid)
mask = SC + SP <= 100
Z3 = np.where(mask, PnL(r_opt, SC, SP), np.nan)

fig.add_trace(go.Surface(x=grid, y=grid, z=Z3,
    colorscale="Plasma", showscale=False, name="SC vs SP"), row=1, col=3)

fig.update_xaxes(title_text="Research %", row=1, col=1)
fig.update_yaxes(title_text="Scale %", row=1, col=1)
fig.update_scenes(xaxis_title="Research %", yaxis_title="Scale %", zaxis_title="PnL", row=1, col=1)

fig.update_scenes(xaxis_title="Research %", yaxis_title="Speed %", zaxis_title="PnL", row=1, col=2)
fig.update_scenes(xaxis_title="Scale %", yaxis_title="Speed %", zaxis_title="PnL", row=1, col=3)

fig.update_layout(title="PnL surface — one parameter fixed at optimum", height=500)
fig.show()

# Speed distribution

Rank-based across all players:
    - Highest speed investment receives a `0.9` multiplier.
    - Lowest receives `0.1`.
    - Everyone in between is scaled linearly by rank, equal investments share the same rank.


### 1/ Nash equilibrium (correct game-theoretic answer)

In [33]:
def speed_real(my_sp, others_sp):
    """
    my_sp   : float, your speed allocation (0–100)
    others_sp: list of floats, other players' speed allocations
    
    Returns your hit rate multiplier in [0.1, 0.9].
    Equal investments share the same rank (dense rank).
    """
    all_sp = [my_sp] + list(others_sp)
    n = len(all_sp)
    
    # dense rank: higher spend = lower rank number (rank 1 = best)
    sorted_unique = sorted(set(all_sp), reverse=True)
    dense_rank = {v: i+1 for i, v in enumerate(sorted_unique)}
    my_rank = dense_rank[my_sp]
    worst_rank = dense_rank[min(all_sp)]
    
    if worst_rank == 1:
        return 0.9  # everyone tied at the top
    
    # linear interpolation: rank 1 → 0.9, worst_rank → 0.1
    return 0.9 - (my_rank - 1) / (worst_rank - 1) * 0.8

def PnL_real(r_pct, sc_pct, sp_pct, others_sp):
    total_alloc = r_pct + sc_pct + sp_pct
    budget_used = BUDGET * (total_alloc / 100)
    gross = research(r_pct) * scale(sc_pct) * speed_real(sp_pct, others_sp)
    return gross - budget_used

def best_response(others_sp, others_r=None, others_sc=None):
    """
    Best response over ALL three variables, not just speed.
    others_r / others_sc not used in PnL (only others_sp affects rank)
    but we free up R and SC to re-optimize given the speed competition.
    """
    def obj(x):
        return -PnL_real(x[0], x[1], x[2], others_sp)
    
    res = minimize(
        obj,
        x0=[16.02, 48.24, 35.74],
        method="SLSQP",
        bounds=[(0,100), (0,100), (0,100)],
        constraints=[{"type":"ineq","fun": lambda x: 100 - sum(x)}],
        options={"ftol":1e-12,"maxiter":10_000}
    )
    return res.x  # returns (r, sc, sp)

def find_nash(n_players=10, tol=1e-3, max_iter=100):
    # each player's strategy is now a full (r, sc, sp) triple
    strategies = np.tile([16.02, 48.24, 35.74], (n_players, 1))
    # start dispersed on speed only, R/SC at known good values
    strategies[:, 2] = np.linspace(0, 35, n_players)

    for iteration in range(max_iter):
        new_strategies = np.zeros_like(strategies)
        for i in range(n_players):
            others_sp = np.delete(strategies[:, 2], i)
            new_strategies[i] = best_response(others_sp)
        
        delta = np.max(np.abs(new_strategies - strategies))
        print(f"iter {iteration:3d} | "
              f"R={new_strategies[:,0].mean():.2f}% "
              f"SC={new_strategies[:,1].mean():.2f}% "
              f"SP={new_strategies[:,2].mean():.2f}% | delta: {delta:.4f}")
        
        if delta < tol:
            print(f"Nash equilibrium found in {iteration+1} iterations")
            break
        strategies = new_strategies
    else:
        print("Warning: did not converge")
    
    return strategies

nash = find_nash()
print(f"\nNash equilibrium:")
print(f"  Research : {nash[:,0].mean():.2f}% ± {nash[:,0].std():.3f}")
print(f"  Scale    : {nash[:,1].mean():.2f}% ± {nash[:,1].std():.3f}")
print(f"  Speed    : {nash[:,2].mean():.2f}% ± {nash[:,2].std():.3f}")

iter   0 | R=16.56% SC=49.99% SP=33.44% | delta: 35.0000
iter   1 | R=16.33% SC=48.84% SP=35.00% | delta: 7.7781
iter   2 | R=16.25% SC=48.85% SP=35.00% | delta: 0.7767
iter   3 | R=16.17% SC=48.83% SP=35.00% | delta: 0.7767
iter   4 | R=16.17% SC=48.83% SP=35.00% | delta: 0.0000
Nash equilibrium found in 5 iterations

Nash equilibrium:
  Research : 16.17% ± 0.000
  Scale    : 48.83% ± 0.000
  Speed    : 35.00% ± 0.000


iter   0 | R=16.10% SC=49.90% SP=33.00% | delta: 35.0000
iter   1 | R=16.00% SC=48.00% SP=35.00% | delta: 8.0000
iter   2 | R=16.00% SC=48.00% SP=35.00% | delta: 0.0000
Nash equilibrium found in 3 iterations

Nash equilibrium:
  Research : 16.00% ± 0.000
  Scale    : 48.00% ± 0.000
  Speed    : 35.00% ± 0.000


In [ ]:
nash_sp = nash_speeds.mean()  # scalar: what everyone else plays
r_fixed, sc_fixed = 16.02, 48.24

my_sp_grid = np.linspace(0, 100 - r_fixed - sc_fixed, 500)
my_pnl     = [PnL_real(r_fixed, sc_fixed, sp, others_sp=[nash_sp] * 9) for sp in my_sp_grid]
naive_pnl  = [PnL(r_fixed, sc_fixed, sp) for sp in my_sp_grid]  # linear proxy for comparison

fig = go.Figure()
fig.add_trace(go.Scatter(x=my_sp_grid, y=my_pnl,   name="PnL vs Nash opponents", line=dict(color="#1D9E75")))
fig.add_trace(go.Scatter(x=my_sp_grid, y=naive_pnl, name="PnL linear proxy",      line=dict(color="#888", dash="dash")))
fig.add_vline(x=nash_sp, line_dash="dot", line_color="#E24B4A", annotation_text=f"Nash sp={nash_sp:.1f}%")
fig.update_layout(
    title="Your PnL vs your speed allocation (opponents at Nash equilibrium)",
    xaxis_title="Your speed %",
    yaxis_title="Net PnL",
    height=450
)
fig.show()

In [29]:
nash_sp = 33.33
r_fixed, sc_fixed = 16.02, 48.24
n_opponents = 9

# sweep your speed, opponents all at nash_sp
my_sp_grid = np.linspace(0, 100 - r_fixed - sc_fixed, 2000)
my_pnl = [
    PnL_real(r_fixed, sc_fixed, sp, others_sp=[nash_sp] * n_opponents)
    for sp in my_sp_grid
]

# also sweep: what if you assume opponents randomize around nash_sp ± sigma?
sigma = 5
n_samples = 300
my_pnl_robust = []
for sp in my_sp_grid:
    samples = [
        PnL_real(r_fixed, sc_fixed, sp,
                 others_sp=np.clip(np.random.normal(nash_sp, sigma, n_opponents), 0, 100))
        for _ in range(n_samples)
    ]
    my_pnl_robust.append(np.mean(samples))

fig = go.Figure()
fig.add_trace(go.Scatter(x=my_sp_grid, y=my_pnl,
    name="opponents fixed at Nash", line=dict(color="#1D9E75")))
fig.add_trace(go.Scatter(x=my_sp_grid, y=my_pnl_robust,
    name=f"opponents ~ N({nash_sp}, {sigma})", line=dict(color="#378ADD")))
fig.add_vline(x=nash_sp, line_dash="dot", line_color="#E24B4A",
    annotation_text=f"Nash sp={nash_sp:.1f}%")
fig.update_layout(
    title="Deviation from Nash: is it worth going higher?",
    xaxis_title="Your speed %", yaxis_title="Net PnL", height=450
)
fig.show()

### 2/ expected PnL over a distribution of opponents (robust)


In [ ]:
### 1/ expected PnL over a distribution of opponents (robust)


### 3/ Monte Carlo simulations on the distribution of resource from over people